# Imports

In [2]:
import random
import math
import os
from pathlib import Path

import gc

import pandas as pd
import numpy as np

from datasets import load_dataset, DatasetDict

from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, SFTConfig

from transformers.modeling_outputs import CausalLMOutput

import torch
import torch.nn as nn

from transformers import PreTrainedTokenizerFast
from model.first_aid_advisor import FirstAidAdvisorLM

/opt/anaconda3/envs/LLM/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Use GPU

In [3]:
# Device selection (GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"  # choose device
print("Using device:", device)

Using device: cpu


# Reproducable Workflow

In [4]:
# Sets random seeds for reproducibility
def set_seed(seed):
    # Set seed for PyTorch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        # Set seed for all available GPUs
        torch.cuda.manual_seed_all(seed)

    # Set seed for NumPy
    np.random.seed(seed)

    # Set seed for Python's built-in random module
    random.seed(seed)

    # Enable deterministic algorithms for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Use deterministic algorithms where available
    torch.use_deterministic_algorithms(True)

# Set random seeds
SEED = 42
set_seed(SEED)

# Setup Files

In [5]:
# Get project root
ROOT_DIR = Path(os.getcwd()).resolve().parent
print("Project root:", ROOT_DIR)

Project root: /Users/seangupta/Desktop/llm


# Helper Functions

In [6]:
# helper functions

def count_trainable_params(model):
    """Return trainable params, total params, and trainable percentage."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    pct = 100 * trainable / total
    return trainable, total, pct


def summarize_trainable_params(model, label="model"):
    """Print and return a compact parameter summary for easy comparison."""
    trainable, total, pct = count_trainable_params(model)
    print(f"{label}: trainable={trainable:,} | total={total:,} | trainable%={pct:.2f}")
    return {
        "trainable_params": trainable,
        "total_params": total,
        "trainable_pct": pct,
    }


def history_df(trainer):
    """Turn a Trainer/SFTTrainer log history into a DataFrame."""
    return pd.DataFrame(trainer.state.log_history)


def epoch_history_table(trainer, columns=None):
    """Show only the most useful training/eval columns."""
    hist = history_df(trainer)
    if "eval_loss" in hist.columns:
        hist["perplexity"] = hist["eval_loss"].apply(
            lambda x: math.exp(x) if pd.notna(x) else None
        )
    if hist.empty:
        return hist
    if columns is None:
        columns = [
            "epoch", "loss", "eval_loss", "eval_accuracy", "eval_macro_f1",
            "learning_rate", "grad_norm", "perplexity"
        ]
    cols = [c for c in columns if c in hist.columns]
    if not cols:
        return hist
    metric_cols = [c for c in cols if c != "epoch"]
    table = hist.dropna(subset=metric_cols, how="all").copy()
    if table.empty:
        table = hist.copy()
    return table[cols].reset_index(drop=True)


def last_logged_metrics(trainer, prefix="eval_"):
    """Grab the last logged evaluation metrics from the trainer history."""
    hist = history_df(trainer)
    if hist.empty:
        return {}
    cols = [c for c in hist.columns if c.startswith(prefix)]
    if not cols:
        return {}
    rows = hist.dropna(subset=cols, how="all")
    if rows.empty:
        return {}
    last = rows.iloc[-1]
    return {c: last[c] for c in cols if pd.notna(last[c])}


def remove_notebook_progress_callback(trainer):
    """Avoid notebook callback issues before calling predict()."""
    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except Exception:
        pass


def cleanup(*objs):
    """Free memory between large runs."""
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Load Dataset

In [7]:
# Config
DATASET_PATH = f"{ROOT_DIR}/data_raw/fine_tuning_data/fine_tuning_dataset.jsonl"

SFT_MAX_TRAIN = 2460
SFT_MAX_VAL = 308
SFT_MAX_TEST = 308

In [8]:
raw_sft = load_dataset("json", data_files=DATASET_PATH, split="train")
print(raw_sft)
print(raw_sft[0])

Generating train split: 3076 examples [00:00, 423909.29 examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 3076
})
{'instruction': 'Provide first-aid guidance for a mild tension headache.', 'input': 'I have a dull headache after working on my computer all day. What should I do?', 'output': 'Try taking an over-the-counter pain reliever like acetaminophen to help ease the discomfort. Rest your eyes, hydrate with water, and take breaks from screens. Follow the medication label instructions and seek care if the headache becomes severe or persistent.'}


In [9]:
sft_shuffled = raw_sft.shuffle(seed=SEED)
train_end = min(SFT_MAX_TRAIN, len(sft_shuffled))
val_end = min(train_end + SFT_MAX_VAL, len(sft_shuffled))
test_end = min(val_end + SFT_MAX_TEST, len(sft_shuffled))

sft_ds = DatasetDict({
    "train": sft_shuffled.select(range(0, train_end)),
    "validation": sft_shuffled.select(range(train_end, val_end)),
    "test": sft_shuffled.select(range(val_end, test_end)),
})

print(sft_ds)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2460
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 308
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 308
    })
})


In [10]:
# Format prompt-completion pairs for SFTTrainer.
# The prompt contains the instruction (and input if present).
# The completion is the target response the model should learn to generate.

def alpaca_prompt(example):
    instruction = example["instruction"].strip()
    input_text = example["input"].strip()
    if input_text:
        return f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
    return f"""### Instruction:
{instruction}

### Response:
"""


def to_prompt_completion(example):
    return {
        "prompt": alpaca_prompt(example),
        "completion": example["output"].strip(),
    }

sft_pc = sft_ds.map(to_prompt_completion, remove_columns=sft_ds["train"].column_names)
print(sft_pc["train"][0])
print()
print(sft_pc["train"][1])

Map: 100%|██████████| 308/308 [00:00<00:00, 32036.64 examples/s]

{'prompt': '### Instruction:\nProvide diarrhea care without medication.\n\n### Input:\nI have mild diarrhea since this morning. What should I do?\n\n### Response:\n', 'completion': 'Drink plenty of fluids to stay hydrated, especially water or electrolyte drinks. Eat simple foods like rice or toast if you feel hungry. Rest and avoid greasy or spicy meals. If it persists or worsens, seek care and follow safety guidance.'}

{'prompt': '### Instruction:\nExplain how to manage a low-grade fever at home.\n\n### Input:\nI have a slight fever and feel tired. What should I do?\n\n### Response:\n', 'completion': 'Rest and drink fluids like water or clear broth to stay hydrated. Keep your clothing light and avoid overheating. Acetaminophen may help with discomfort if needed. Get medical advice if the fever continues or gets higher.'}


# LoRA Fine Tuning

In [ ]:
# Config
SFT_MAX_LENGTH = 512

SFT_LORA_R = 8
SFT_LORA_ALPHA = 16
SFT_LORA_DROPOUT = 0.05
SFT_LORA_TARGETS = ["Wq", "Wk", "Wv"]

SFT_EPOCHS = 2

CHECKPOINTS_OUTPUT_DIR = f"{ROOT_DIR}/code/lora_fine_tuning_checkpoints"
REPORTS_OUTPUT_DIR = f"{ROOT_DIR}/results"

In [12]:
model_obj = FirstAidAdvisorLM(model_name="pretrained", print_details=True) # Load pretrained model
model = model_obj.model
tokenizer = model_obj.tokenizer

model.train()

Tokenizer_vocab_size: 24000
vocab_size: 24000
context_length: 512
emb_dim: 512
n_heads: 8
n_layers: 8
drop_rate: 0.1
qkv_bias: True
model_type: gpt2
_attn_implementation: eager
torch_dtype: torch.float32


```
MyCausalLMWrapper(
  (model): GPTModel(
    (tok_emb): Embedding(24000, 512)
    (pos_emb): Embedding(512, 512)
    (drop_emb): Dropout(p=0.1, inplace=False)
    (blocks): ModuleList(
      (0-7): 8 x TransformerBlock(
        (ln1): LayerNorm()
        (ln2): LayerNorm()
        (attn): MultiHeadCausalSelfAttention(
          (Wq): Linear(in_features=512, out_features=512, bias=True)
          (Wk): Linear(in_features=512, out_features=512, bias=True)
          (Wv): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
          (attn_drop): Dropout(p=0.1, inplace=False)
        )
        (ff): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inplace=False)
          )
        )
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
    )
    (final_ln): LayerNorm()
    (out_head): Linear(in_features=512, out_features=24000, bias=False)
  )
)
```

MyCausalLMWrapper(
  (model): GPTModel(
    (tok_emb): Embedding(24000, 512)
    (pos_emb): Embedding(512, 512)
    (drop_emb): Dropout(p=0.1, inplace=False)
    (blocks): ModuleList(
      (0-7): 8 x TransformerBlock(
        (ln1): LayerNorm()
        (ln2): LayerNorm()
        (attn): MultiHeadCausalSelfAttention(
          (Wq): Linear(in_features=512, out_features=512, bias=True)
          (Wk): Linear(in_features=512, out_features=512, bias=True)
          (Wv): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
          (attn_drop): Dropout(p=0.1, inplace=False)
        )
        (ff): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=2048, out_features=512, bias=True)
            (3): Dropout(p=0.1, inplace=False)
          )
        )
        (resid_drop): Dropo

In [13]:
# LoRA config for causal LM fine-tuning.
sft_lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=SFT_LORA_R,
    lora_alpha=SFT_LORA_ALPHA,
    lora_dropout=SFT_LORA_DROPOUT,
    target_modules=SFT_LORA_TARGETS,
    bias="none"
)

In [14]:
# Quick parameter check on a wrapped copy, then free the probe model.
sft_lora_probe = get_peft_model(FirstAidAdvisorLM(model_name='pretrained').model, sft_lora_config)
sft_lora_params_reprot = summarize_trainable_params(
    sft_lora_probe,
    label="Instruction SFT / LoRA (parameter check only)"
)
cleanup(sft_lora_probe)

for key, value in sft_lora_params_reprot.items():
    print(f"{key}: {value}")

Instruction SFT / LoRA (parameter check only): trainable=196,608 | total=50,254,848 | trainable%=0.39
trainable_params: 196608
total_params: 50254848
trainable_pct: 0.39122195733235526


In [15]:
sft_lora_args = SFTConfig(
    output_dir=CHECKPOINTS_OUTPUT_DIR,
    num_train_epochs=SFT_EPOCHS,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=20,
    logging_strategy="steps",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    disable_tqdm=False,
    max_length=SFT_MAX_LENGTH,
    completion_only_loss=True,
    gradient_checkpointing=False,
    seed=SEED,
)

# Wrap it in the 'Fast' wrapper
fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    eos_token="<|endoftext|>",
    pad_token="<|endoftext|>"
)

# Let SFTTrainer wrap the model with LoRA through peft_config.
sft_lora_trainer = SFTTrainer(
    model=model,
    args=sft_lora_args,
    train_dataset=sft_pc["train"],
    eval_dataset=sft_pc["validation"],
    processing_class=fast_tokenizer,
    peft_config=sft_lora_config,
)

Tokenizing eval dataset: 100%|██████████| 308/308 [00:00<00:00, 5203.43 examples/s]


In [16]:
# Recompute params on the actual wrapped model used by the trainer.
sft_lora_params_reprot = summarize_trainable_params(sft_lora_trainer.model, label="Instruction SFT / LoRA")
for key, value in sft_lora_params_reprot.items():
    print(f"{key}: {value}")

Instruction SFT / LoRA: trainable=196,608 | total=50,254,848 | trainable%=0.39
trainable_params: 196608
total_params: 50254848
trainable_pct: 0.39122195733235526


In [17]:
sft_lora_train_output = sft_lora_trainer.train()
remove_notebook_progress_callback(sft_lora_trainer)

sft_lora_eval = last_logged_metrics(sft_lora_trainer)
sft_lora_params_reprot.update({
    "train_runtime": sft_lora_train_output.metrics.get("train_runtime"),
    **sft_lora_eval,
})

display(epoch_history_table(sft_lora_trainer, ["epoch", "loss", "eval_loss", "perplexity", "learning_rate", "grad_norm"]))

display(pd.DataFrame([sft_lora_params_reprot])[[
    "trainable_params", "total_params", "trainable_pct", "train_runtime", "eval_loss"
]].round(4))

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 0, 'pad_token_id': 0}.
/opt/anaconda3/envs/LLM/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,39.096655,9.619567
2,37.689975,9.288036


/opt/anaconda3/envs/LLM/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


,epoch,loss,eval_loss,perplexity,learning_rate,grad_norm
0,0.065041,42.364459,NaN,NaN,9.691558e-06,1.989820
1,0.130081,41.720370,NaN,NaN,9.366883e-06,1.627170
2,0.195122,41.990738,NaN,NaN,9.042208e-06,2.193183
3,0.260163,41.582956,NaN,NaN,8.717532e-06,3.148002
4,0.325203,40.968857,NaN,NaN,8.392857e-06,1.531130
5,0.390244,41.227014,NaN,NaN,8.068182e-06,2.562056
6,0.455285,40.554034,NaN,NaN,7.743506e-06,1.792340
7,0.520325,40.587457,NaN,NaN,7.418831e-06,2.937410
8,0.585366,40.077390,NaN,NaN,7.094156e-06,1.962532
9,0.650407,39.989175,NaN,NaN,6.769481e-06,1.546661


,trainable_params,total_params,trainable_pct,train_runtime,eval_loss
0,196608,50254848,0.3912,209.0581,9.288


# Report data

In [18]:
sft_lora_info_df = pd.DataFrame([sft_lora_params_reprot])[[
    "trainable_params", "total_params", "trainable_pct", "train_runtime", "eval_loss"
]].round(4)

sft_lora_info_md = sft_lora_info_df.to_markdown(index=False)

print(sft_lora_info_md)

with open(f"{ROOT_DIR}/results/lora_training_info_table.md", "w") as f:
    f.write(sft_lora_info_md)

|   trainable_params |   total_params |   trainable_pct |   train_runtime |   eval_loss |
|-------------------:|---------------:|----------------:|----------------:|------------:|
|             196608 |    5.02548e+07 |          0.3912 |         209.058 |       9.288 |


In [19]:
sft_lora_trainer_df = epoch_history_table(sft_lora_trainer, ["epoch", "loss", "eval_loss", "perplexity", "learning_rate", "grad_norm"])

sft_lora_trainer_md = sft_lora_trainer_df.to_markdown(index=False)

print(sft_lora_trainer_md)

with open(f"{ROOT_DIR}/results/lora_training_history_table.md", "w") as f:
    f.write(sft_lora_trainer_md)

|     epoch |     loss |   eval_loss |   perplexity |   learning_rate |   grad_norm |
|----------:|---------:|------------:|-------------:|----------------:|------------:|
| 0.0650407 |  42.3645 |   nan       |        nan   |     9.69156e-06 |     1.98982 |
| 0.130081  |  41.7204 |   nan       |        nan   |     9.36688e-06 |     1.62717 |
| 0.195122  |  41.9907 |   nan       |        nan   |     9.04221e-06 |     2.19318 |
| 0.260163  |  41.583  |   nan       |        nan   |     8.71753e-06 |     3.148   |
| 0.325203  |  40.9689 |   nan       |        nan   |     8.39286e-06 |     1.53113 |
| 0.390244  |  41.227  |   nan       |        nan   |     8.06818e-06 |     2.56206 |
| 0.455285  |  40.554  |   nan       |        nan   |     7.74351e-06 |     1.79234 |
| 0.520325  |  40.5875 |   nan       |        nan   |     7.41883e-06 |     2.93741 |
| 0.585366  |  40.0774 |   nan       |        nan   |     7.09416e-06 |     1.96253 |
| 0.650407  |  39.9892 |   nan       |        nan   | 

# Save model

In [ ]:
SAVE_LORA_ADAPTER_FILEPATH = f"{ROOT_DIR}/code/model/model_weights/sft_lora_adapter"
sft_lora_trainer.model.save_pretrained(SAVE_LORA_ADAPTER_FILEPATH)